In [2]:
import duckdb
import time
import glob

In [2]:
def ingest_parquet_to_postgres(parquet_file: str
                               , table_name: str
                               ):
    DB_CONNECTION = "dbname=taxi_rides_ny user=admin password=admin1234 host=localhost port=5432"
    TABLE_NAME = f"{table_name}"
    SCHEMA_NAME = "public"

    con = duckdb.connect(database=':memory:')

    con.execute("INSTALL postgres;")
    con.execute("LOAD postgres;")

    con.execute(f"ATTACH '{DB_CONNECTION}' AS pg_db (TYPE POSTGRES);")

    con.execute(f"""
        CREATE TABLE IF NOT EXISTS pg_db.{SCHEMA_NAME}.{TABLE_NAME} AS 
        SELECT * FROM '{parquet_file}' LIMIT 0;
    """)

    con.execute(f"""
        INSERT INTO pg_db.{SCHEMA_NAME}.{TABLE_NAME} 
        SELECT * FROM '{parquet_file}';
    """)

    con.close()

In [16]:
# Configuration
DB_CONNECTION = "dbname=taxi_rides_ny user=admin password=admin1234 host=localhost port=5432"
FOLDER_PATH = "./data/green/*.parquet" # <-- Note the wildcard '*'
TABLE_NAME = "green_tripdata"
SCHEMA_NAME = "public"

def ingest_folder():
    start_time = time.time()
    
    # 1. Setup DuckDB & Postgres Link
    con = duckdb.connect(database=':memory:')
    con.execute("INSTALL postgres; LOAD postgres;")
    con.execute(f"ATTACH '{DB_CONNECTION}' AS pg_db (TYPE POSTGRES);")

    print(f"--- 1. Introspecting all files in {FOLDER_PATH} ---")
    
    # 2. Get the Unified Schema
    # DuckDB's 'read_parquet' can take a glob pattern (e.g., data/*.parquet)
    # It automatically merges schemas from all files!
    schema_query = f"DESCRIBE SELECT * FROM read_parquet('{FOLDER_PATH}')"
    schema_info = con.sql(schema_query).fetchall()
    
    # 3. Build the Dynamic SQL (Normalization Logic)
    select_clauses = []
    print("--- Detected Unified Columns ---")
    for col in schema_info:
        original_name = col[0]
        
        # Normalize: Lowercase, replace space/dash with underscore
        clean_name = original_name.lower().replace(' ', '_').replace('-', '_')
        
        # Quote original name, Alias clean name
        select_clauses.append(f'"{original_name}" AS "{clean_name}"')
        
        # Optional: Print only the first few to avoid spamming console
        if len(select_clauses) <= 5:
            print(f"   > {original_name} -> {clean_name}")

    if len(select_clauses) > 5: print(f"   > ... and {len(select_clauses)-5} more.")

    final_select_list = ", ".join(select_clauses)

    # 4. Create & Load in One Shot
    # We use 'union_by_name=True' to ensure that if file A has column X and file B doesn't,
    # it fills with NULLs instead of crashing.
    print(f"--- 2. Batch Loading into Postgres ({TABLE_NAME}) ---")
    
    # Drop previous table to start fresh (for idempotency)
    con.execute(f"DROP TABLE IF EXISTS pg_db.{SCHEMA_NAME}.{TABLE_NAME}")
    
    load_query = f"""
        CREATE TABLE pg_db.{SCHEMA_NAME}.{TABLE_NAME} AS 
        SELECT {final_select_list} 
        FROM read_parquet('{FOLDER_PATH}', union_by_name=True);
    """
    
    con.execute(load_query)
    
    end_time = time.time()
    print(f"--- Success! Batch loaded in {end_time - start_time:.2f} seconds ---")

if __name__ == "__main__":
    # Safety Check: Ensure files exist before running
    if not glob.glob(FOLDER_PATH):
        print(f"Error: No files found at {FOLDER_PATH}")
    else:
        ingest_folder()

--- 1. Introspecting all files in ./data/green/*.parquet ---
--- Detected Unified Columns ---
   > VendorID -> vendorid
   > lpep_pickup_datetime -> lpep_pickup_datetime
   > lpep_dropoff_datetime -> lpep_dropoff_datetime
   > store_and_fwd_flag -> store_and_fwd_flag
   > RatecodeID -> ratecodeid
   > ... and 15 more.
--- 2. Batch Loading into Postgres (green_tripdata) ---
--- Success! Batch loaded in 18.94 seconds ---


In [17]:
# Configuration
DB_CONNECTION = "dbname=taxi_rides_ny user=admin password=admin1234 host=localhost port=5432"
FOLDER_PATH = "./data/yellow/*.parquet" # <-- Note the wildcard '*'
TABLE_NAME = "yellow_tripdata"
SCHEMA_NAME = "public"

def ingest_folder():
    start_time = time.time()
    
    # 1. Setup DuckDB & Postgres Link
    con = duckdb.connect(database=':memory:')
    con.execute("INSTALL postgres; LOAD postgres;")
    con.execute(f"ATTACH '{DB_CONNECTION}' AS pg_db (TYPE POSTGRES);")

    print(f"--- 1. Introspecting all files in {FOLDER_PATH} ---")
    
    # 2. Get the Unified Schema
    # DuckDB's 'read_parquet' can take a glob pattern (e.g., data/*.parquet)
    # It automatically merges schemas from all files!
    schema_query = f"DESCRIBE SELECT * FROM read_parquet('{FOLDER_PATH}')"
    schema_info = con.sql(schema_query).fetchall()
    
    # 3. Build the Dynamic SQL (Normalization Logic)
    select_clauses = []
    print("--- Detected Unified Columns ---")
    for col in schema_info:
        original_name = col[0]
        
        # Normalize: Lowercase, replace space/dash with underscore
        clean_name = original_name.lower().replace(' ', '_').replace('-', '_')
        
        # Quote original name, Alias clean name
        select_clauses.append(f'"{original_name}" AS "{clean_name}"')
        
        # Optional: Print only the first few to avoid spamming console
        if len(select_clauses) <= 5:
            print(f"   > {original_name} -> {clean_name}")

    if len(select_clauses) > 5: print(f"   > ... and {len(select_clauses)-5} more.")

    final_select_list = ", ".join(select_clauses)

    # 4. Create & Load in One Shot
    # We use 'union_by_name=True' to ensure that if file A has column X and file B doesn't,
    # it fills with NULLs instead of crashing.
    print(f"--- 2. Batch Loading into Postgres ({TABLE_NAME}) ---")
    
    # Drop previous table to start fresh (for idempotency)
    con.execute(f"DROP TABLE IF EXISTS pg_db.{SCHEMA_NAME}.{TABLE_NAME}")
    
    load_query = f"""
        CREATE TABLE pg_db.{SCHEMA_NAME}.{TABLE_NAME} AS 
        SELECT {final_select_list} 
        FROM read_parquet('{FOLDER_PATH}', union_by_name=True);
    """
    
    con.execute(load_query)
    
    end_time = time.time()
    print(f"--- Success! Batch loaded in {end_time - start_time:.2f} seconds ---")

if __name__ == "__main__":
    # Safety Check: Ensure files exist before running
    if not glob.glob(FOLDER_PATH):
        print(f"Error: No files found at {FOLDER_PATH}")
    else:
        ingest_folder()

--- 1. Introspecting all files in ./data/yellow/*.parquet ---
--- Detected Unified Columns ---
   > VendorID -> vendorid
   > tpep_pickup_datetime -> tpep_pickup_datetime
   > tpep_dropoff_datetime -> tpep_dropoff_datetime
   > passenger_count -> passenger_count
   > trip_distance -> trip_distance
   > ... and 13 more.
--- 2. Batch Loading into Postgres (yellow_tripdata) ---
--- Success! Batch loaded in 286.32 seconds ---


In [3]:
# Configuration
DB_CONNECTION = "dbname=taxi_rides_ny user=admin password=admin1234 host=localhost port=5432"
FOLDER_PATH = "./data/fhv/*.parquet" # <-- Note the wildcard '*'
TABLE_NAME = "fhv_tripdata"
SCHEMA_NAME = "public"

def ingest_folder():
    start_time = time.time()
    
    # 1. Setup DuckDB & Postgres Link
    con = duckdb.connect(database=':memory:')
    con.execute("INSTALL postgres; LOAD postgres;")
    con.execute(f"ATTACH '{DB_CONNECTION}' AS pg_db (TYPE POSTGRES);")

    print(f"--- 1. Introspecting all files in {FOLDER_PATH} ---")
    
    # 2. Get the Unified Schema
    # DuckDB's 'read_parquet' can take a glob pattern (e.g., data/*.parquet)
    # It automatically merges schemas from all files!
    schema_query = f"DESCRIBE SELECT * FROM read_parquet('{FOLDER_PATH}')"
    schema_info = con.sql(schema_query).fetchall()
    
    # 3. Build the Dynamic SQL (Normalization Logic)
    select_clauses = []
    print("--- Detected Unified Columns ---")
    for col in schema_info:
        original_name = col[0]
        
        # Normalize: Lowercase, replace space/dash with underscore
        clean_name = original_name.lower().replace(' ', '_').replace('-', '_')
        
        # Quote original name, Alias clean name
        select_clauses.append(f'"{original_name}" AS "{clean_name}"')
        
        # Optional: Print only the first few to avoid spamming console
        if len(select_clauses) <= 5:
            print(f"   > {original_name} -> {clean_name}")

    if len(select_clauses) > 5: print(f"   > ... and {len(select_clauses)-5} more.")

    final_select_list = ", ".join(select_clauses)

    # 4. Create & Load in One Shot
    # We use 'union_by_name=True' to ensure that if file A has column X and file B doesn't,
    # it fills with NULLs instead of crashing.
    print(f"--- 2. Batch Loading into Postgres ({TABLE_NAME}) ---")
    
    # Drop previous table to start fresh (for idempotency)
    con.execute(f"DROP TABLE IF EXISTS pg_db.{SCHEMA_NAME}.{TABLE_NAME}")
    
    load_query = f"""
        CREATE TABLE pg_db.{SCHEMA_NAME}.{TABLE_NAME} AS 
        SELECT {final_select_list} 
        FROM read_parquet('{FOLDER_PATH}', union_by_name=True);
    """
    
    con.execute(load_query)
    
    end_time = time.time()
    print(f"--- Success! Batch loaded in {end_time - start_time:.2f} seconds ---")

if __name__ == "__main__":
    # Safety Check: Ensure files exist before running
    if not glob.glob(FOLDER_PATH):
        print(f"Error: No files found at {FOLDER_PATH}")
    else:
        ingest_folder()

--- 1. Introspecting all files in ./data/fhv/*.parquet ---
--- Detected Unified Columns ---
   > dispatching_base_num -> dispatching_base_num
   > pickup_datetime -> pickup_datetime
   > dropOff_datetime -> dropoff_datetime
   > PUlocationID -> pulocationid
   > DOlocationID -> dolocationid
   > ... and 2 more.
--- 2. Batch Loading into Postgres (fhv_tripdata) ---
--- Success! Batch loaded in 63.32 seconds ---


In [13]:
green_tripdata = glob.glob("./data/green/*.parquet")
yellow_tripdata = glob.glob("./data/yellow/*.parquet")

In [12]:
for file in sorted(green_tripdata):
    ingest_parquet_to_postgres(parquet_file=file, table_name="green_tripdata")

In [14]:
for file in sorted(yellow_tripdata):
    ingest_parquet_to_postgres(parquet_file=file, table_name="yellow_tripdata")

In [ ]:
import duckdb
import time

# 1. Configuration
DB_CONNECTION = "dbname=analytics_db user=admin password=password host=localhost port=5432"
PARQUET_FILE = "./data/my_source_data.parquet"
TABLE_NAME = "raw_orders"
SCHEMA_NAME = "public"

def ingest_with_duckdb():
    start_time = time.time()
    print(f"--- Starting DuckDB Ingestion for {PARQUET_FILE} ---")

    # 2. Initialize DuckDB (In-memory)
    # We don't need a file for DuckDB itself, so we use ':memory:'
    con = duckdb.connect(database=':memory:')

    # 3. Install & Load the Postgres Extension
    # This allows DuckDB to "talk" natively to Postgres
    con.execute("INSTALL postgres;")
    con.execute("LOAD postgres;")

    # 4. Attach Postgres
    # This treats your Postgres DB as if it were just another DuckDB file
    con.execute(f"ATTACH '{DB_CONNECTION}' AS pg_db (TYPE POSTGRES);")

    # 5. The "Magic" Move
    # We use valid SQL to select from the file and insert directly into the attached DB.
    # DuckDB handles the type conversion and batching automatically.
    
    # First, ensure table exists (Optional - usually done by dbt or specific DDL)
    # But for raw loading, we can create it on the fly if needed:
    con.execute(f"""
        CREATE TABLE IF NOT EXISTS pg_db.{SCHEMA_NAME}.{TABLE_NAME} AS 
        SELECT * FROM '{PARQUET_FILE}' LIMIT 0;
    """)
    
    # Perform the Insert
    print("Executing Insert...")
    con.execute(f"""
        INSERT INTO pg_db.{SCHEMA_NAME}.{TABLE_NAME} 
        SELECT * FROM '{PARQUET_FILE}';
    """)

    end_time = time.time()
    print(f"--- Success! Loaded in {end_time - start_time:.2f} seconds ---")

if __name__ == "__main__":
    ingest_with_duckdb()